# Notebook 1 — Environment verification & data download

Verifies Spark/local filesystem access, downloads raw datasets, writes files under `hdfs_paths.BASE`, and prints schema previews.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("."))
import hdfs_paths as hp

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("MusicTrend_01_Environment")
    .config("spark.sql.shuffle.partitions", hp.SHUFFLE_PARTITIONS)
    .getOrCreate()
)
sc = spark.sparkContext
print(f"Spark version: {spark.version}")
print(f"Default parallelism: {sc.defaultParallelism}")
print(f"Data root: {hp.BASE}")


In [ ]:
# Filesystem connectivity check (local mode)
print("Current working directory:", os.getcwd())
print("Base path:", hp.BASE)
print("Base exists:", os.path.exists(hp.BASE))


In [ ]:
# Create local directories
for d in [
    f"{hp.BASE}/raw/lastfm",
    f"{hp.BASE}/raw/spotify_charts",
    f"{hp.BASE}/raw/billboard",
    f"{hp.BASE}/raw/msd_audio",
    f"{hp.BASE}/processed/events",
    f"{hp.BASE}/processed/spotify",
    f"{hp.BASE}/processed/billboard",
    f"{hp.BASE}/processed/audio",
    f"{hp.BASE}/processed/features",
    f"{hp.BASE}/streaming/input",
    f"{hp.BASE}/streaming/output",
    f"{hp.BASE}/streaming/checkpoint",
    f"{hp.BASE}/models/rf_model",
]:
    os.makedirs(d, exist_ok=True)
print("Directories ensured.")


## Download datasets

Use direct URLs where possible. **Kaggle** sources may require API token or browser download. This notebook copies local files into the project data folders under `hdfs_paths.BASE`. Document any manual download step in your run log.


In [ ]:
import urllib.request
import zipfile
import shutil

LOCAL = os.path.abspath("_downloads")
os.makedirs(LOCAL, exist_ok=True)

def fetch(url, dest):
    try:
        print("Fetching", url)
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print("Fetch failed:", e)
        return False

def copy_local_to_project_data(local_path, target_path):
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    shutil.copyfile(local_path, target_path)
    print(f"Copied: {local_path} -> {target_path}")

# Last.fm HetRec 2011
lf_zip = os.path.join(LOCAL, "hetrec2011-lastfm-2k.zip")
if fetch("https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip", lf_zip):
    with zipfile.ZipFile(lf_zip, "r") as z:
        names = z.namelist()
        # Archive layout can vary; resolve the file by suffix instead of hardcoded folder.
        member = next((n for n in names if n.lower().endswith("user_artists.dat")), None)
        if member is None:
            raise FileNotFoundError("user_artists.dat not found in Last.fm zip archive")
        z.extract(member, LOCAL)
    copy_local_to_project_data(os.path.join(LOCAL, member), hp.RAW_LASTFM)

# Spotify / Billboard / MSD: usually manual download, then copy from LOCAL if present
for filename, path in [
    ("charts.csv", hp.RAW_SPOTIFY),
    ("Hot 100.csv", hp.RAW_BILLBOARD),
    ("msd_audio_features.csv", hp.RAW_MSD),
]:
    local_file = os.path.join(LOCAL, filename)
    if os.path.exists(local_file):
        copy_local_to_project_data(local_file, path)
    else:
        print(f"Missing local file: {local_file}")
        print(f"Download it manually, place in {LOCAL}, then rerun this cell to copy to {path}")


In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType, DateType,
)

# Explicit schemas — adjust if your CSV headers differ slightly

lastfm_schema = StructType(
    [
        StructField("userID", StringType(), True),
        StructField("artistID", StringType(), True),
        StructField("weight", StringType(), True),
    ]
)

spotify_schema = StructType(
    [
        StructField("title", StringType(), True),
        StructField("rank", IntegerType(), True),
        StructField("date", StringType(), True),
        StructField("artist", StringType(), True),
        StructField("url", StringType(), True),
        StructField("region", StringType(), True),
        StructField("chart", StringType(), True),
        StructField("trend", StringType(), True),
        StructField("streams", LongType(), True),
    ]
)

# Billboard export variants exist — keep string types for flexible parse
billboard_schema = StructType(
    [
        StructField("WeekID", StringType(), True),
        StructField("Song", StringType(), True),
        StructField("Performer", StringType(), True),
        StructField("SongID", StringType(), True),
        StructField("Instance", StringType(), True),
        StructField("Previous Week Position", StringType(), True),
        StructField("Peak Position", StringType(), True),
        StructField("Weeks on Chart", StringType(), True),
    ]
)

msd_schema = StructType(
    [
        StructField("artist_name", StringType(), True),
        StructField("song_title", StringType(), True),
        StructField("tempo", DoubleType(), True),
        StructField("energy", DoubleType(), True),
        StructField("loudness", DoubleType(), True),
        StructField("danceability", DoubleType(), True),
        StructField("key", StringType(), True),
        StructField("mode", StringType(), True),
    ]
)

def preview_csv(path, schema, desc, header=True, sep=","):
    try:
        df = spark.read.schema(schema).option("header", header).option("sep", sep).csv(path)
        print("===", desc, "===")
        df.printSchema()
        df.show(5, truncate=False)
        print("rows:", df.count())
    except Exception as e:
        print(desc, "not available yet:", e)

preview_csv(hp.RAW_LASTFM, lastfm_schema, "Last.fm", header=False, sep="\t")
preview_csv(hp.RAW_SPOTIFY, spotify_schema, "Spotify")
preview_csv(hp.RAW_BILLBOARD, billboard_schema, "Billboard")
preview_csv(hp.RAW_MSD, msd_schema, "MSD audio")


## Outputs confirmed

- Spark session and local filesystem reachable.
- Raw paths defined in `hdfs_paths.py` populated (or documented for manual download + notebook copy).
- Schema previews executed for each dataset.
